<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Notes/09-unsupervised_learning/09-unsupervised_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 9: Unsupervised Learning

**DSC 210 Foundations of Data Science**

References:
- [Hands-on Introduction to Data Science with Python](https://florian-huber.github.io/data_science_course/) (CC BY-NC-SA 4.0)
- [scikit-learn user guide](https://scikit-learn.org/stable/modules/clustering.html), on clustering and decomposition

```
ASK  ->  GET  ->  EXPLORE  ->  [ MODEL ]  ->  COMMUNICATE
```

*Last major revision: 2026-08-13*

Everything so far has been description: what is in the data, what is wrong with it, what it looks like. Today we start **modeling**, and we begin with the half of modeling that has no answer key.

## Key Concepts

- Distinguish **supervised** from **unsupervised** learning by asking whether the data has labels
- State the **k-means objective** and carry out the algorithm by hand on a small dataset
- Explain why k-means needs `k` chosen in advance, and use an elbow plot to choose it
- Build a **dendrogram** by hand with single linkage, and read a cut as a clustering
- Describe how **DBSCAN** differs from k-means, and when its shape assumptions win
- Use **PCA** to reduce dimensions, and explain why features must be scaled first

---
## 1. Learning Without an Answer Key
---

Suppose a store hands you 2,240 customer records and asks: *"what kinds of customers do we have?"*

Notice what is missing from that question. Nobody has told you what the kinds *are*. There is no column labelled `customer_type` that you are trying to predict. There is no answer key against which your output can be graded.

**Definition.** In **supervised learning** the data includes a label (an answer) for each case, and the goal is to predict that label for new cases. In **unsupervised learning** there are no labels, and the goal is to find structure that was already there.

| | Supervised | Unsupervised |
| --- | --- | --- |
| Data has labels? | Yes | No |
| Typical question | "Will this customer respond?" | "What kinds of customers are there?" |
| How do you know you are right? | Compare predictions to held-out labels | You cannot, not directly |
| This course | Module 10 | Today |

When there is no answer key, **you cannot compute an accuracy**. A clustering is not right or wrong. That is uncomfortable, and it is also why the techniques in this module are so often paired with the domain expertise circle from Module 1.

Today's two families:

- **Clustering** groups similar cases together. (Sections 2 through 5.)
- **Dimension reduction** replaces many features with a few that carry most of the information. (Section 6.)

Both halves of that table live inside the same field. It is worth seeing where that field sits before we go further.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/09-unsupervised_learning/fig_ai_vs_ml_vs_deep_learning.png?raw=true" width="520">

*Machine learning is one region of a larger territory, and **both** supervised and unsupervised learning sit inside it. Clustering is not a lesser relative of prediction; it is the other half of the same ring. Deep learning, the innermost circle, is a family of methods we will not reach in this course.*

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/09-unsupervised_learning/fig_clustering_intro.png?raw=true" width="620">

*Clustering asks the machine to find groups that nobody labelled. The same points can be grouped more than one defensible way, which is exactly why judging the result is a human job.*

In [ ]:
# RUN-TOGETHER
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

---
## 2. k-means Clustering
---
### 2.1 The idea, and the objective

k-means splits the data into `k` groups, each represented by a **centroid** (the mean of the points assigned to it). It does this by trying to make every point close to its own centroid.

**Definition.** The **within-cluster sum of squares** (WCSS) of a clustering is

$$\text{WCSS} \;=\; \sum_{j=1}^{k} \; \sum_{x \,\in\, C_j} \left(x - c_j\right)^2$$

where $C_j$ is the $j$-th cluster and $c_j$ is its centroid. **k-means searches for the assignment that makes WCSS as small as possible.**

Read the formula in words before moving on: for each cluster, measure how far every point is from that cluster's centre, square it, and add everything up. A small WCSS means tight clusters.

The algorithm is a repeated two-step:

1. **Assign.** Put each point with the nearest centroid.
2. **Update.** Move each centroid to the mean of the points assigned to it.

Repeat until nothing changes. Each step can only lower WCSS or leave it alone, which is why the procedure stops.

The four panels below show one full run. **A**: centroids are placed. **B**: every point is assigned to its nearest centroid. **C**: each centroid moves to the mean of its members. **D**: the process has stopped changing.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/09-unsupervised_learning/fig_kmeans_sketch.png?raw=true" width="700">

#### **Activity 9.1 - k-means by hand**

Before we run any code, we will compute the algorithm by hand. Consider four points on a line:

$$x = 1,\; 2,\; 6,\; 7 \qquad k = 2$$

We will deliberately start with a **bad** pair of centroids, $c_1 = 1$ and $c_2 = 2$, both sitting in the same clump, to see whether k-means can recover.

Work down the table. For each point, compute the distance to each centroid and assign it to the nearer one. Then recompute each centroid as the mean of its members, and compute the WCSS.

**Iteration 1** ($c_1 = 1$, $c_2 = 2$)

| point | $\lvert x - c_1 \rvert$ | $\lvert x - c_2 \rvert$ | cluster |
| --- | --- | --- | --- |
| 1 |  |  |  |
| 2 |  |  |  |
| 6 |  |  |  |
| 7 |  |  |  |

New $c_1$ = __________  New $c_2$ = __________  WCSS = __________

**Iteration 2** (use your new centroids)

| point | $\lvert x - c_1 \rvert$ | $\lvert x - c_2 \rvert$ | cluster |
| --- | --- | --- | --- |
| 1 |  |  |  |
| 2 |  |  |  |
| 6 |  |  |  |
| 7 |  |  |  |

New $c_1$ = __________  New $c_2$ = __________  WCSS = __________

**Iteration 3** (use your new centroids)

Did any point change cluster? If not, the algorithm has **converged** and you are done.

**Questions.**

**A.** How many iterations did it take? What told you to stop?

**B.** Write down your two WCSS values. What happened to WCSS from iteration 1 to iteration 2, and why could it never have gone up?

**C.** Now redo iteration 1 only, starting instead from $c_1 = 1$ and $c_2 = 7$. How many iterations would this start have needed? What does that tell you about the initial centroids?

#### **Class Example 9.1 - The same four points, in code**

Now we check our arithmetic. `KMeans` from scikit-learn expects a 2-D array, so our four numbers become four rows of one column.

In [ ]:
# RUN-TOGETHER
from sklearn.cluster import KMeans

X = np.array([[1.], [2.], [6.], [7.]])

km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)

print('cluster labels :', km.labels_)
print('centroids      :', km.cluster_centers_.ravel())
print('WCSS (inertia_):', round(km.inertia_, 3))

Compare against your table. The labels may be swapped (cluster "0" and cluster "1" are arbitrary names), but the **centroids** and the **WCSS** should match what you computed by hand.

scikit-learn calls the WCSS `inertia_`. It is the same quantity from the definition above, and we will use it again in Section 3.

One detail worth noting: `n_init=10` runs the whole algorithm ten times from ten different random starts and keeps the best. Activity 9.1 part C is exactly why that argument exists.

Here is the whole run as a picture. Check each panel against the row of your table.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/09-unsupervised_learning/fig_kmeans_by_hand.png?raw=true" width="640">

Notice panel 2 to panel 3: the point at 2 changes sides. That single switch is what drops WCSS from 14 to 1.

### 2.2 k-means on real data: the penguins

Four points on was a nice toy problem. Here is the same algorithm on a dataset we have seen before.

The penguins dataset has four body measurements and, in a separate column, the species. We are going to **hide the species from the algorithm**, cluster on the measurements alone, and then reveal the species to see what k-means found on its own. 

In [ ]:
# RUN-TOGETHER
penguins = sns.load_dataset('penguins').dropna()

features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
sns.pairplot(penguins, vars=features, hue=None, diag_kind='hist')

In [ ]:
# RUN-TOGETHER
from sklearn.preprocessing import StandardScaler

X_pen = StandardScaler().fit_transform(penguins[features])   # scale first!

km_pen = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_pen)
penguins['cluster'] = km_pen.labels_

print('cluster sizes:')
print(penguins['cluster'].value_counts().sort_index())

Now the reveal. Cross-tabulate the clusters against the species the algorithm never saw.

In [ ]:
# RUN-TOGETHER
print(pd.crosstab(penguins['cluster'], penguins['species']))

What can we read from this table?

- One cluster contains **119 Gentoo and nothing else**. k-means found that species perfectly, without being told it existed.
- Another is almost all Adelie; a third is mostly Chinstrap.
- The mistakes are all **Adelie confused with Chinstrap**, in both directions.

About 92% of penguins land in the "right" cluster, and the errors are not random. Adelie and Chinstrap are close in body size and differ mainly in bill length, while Gentoo are substantially bigger than both. So k-means did not find *species*. It found **size**, and size happens to identify Gentoo cleanly while separating the other two only partly.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/09-unsupervised_learning/fig_kmeans_penguins.png?raw=true" width="820">

> **Discuss.** The algorithm was given four measurements and no labels, and it recovered most of the species structure. Does that mean clustering "works"? Now suppose the three species had been identical in size but differed in colour, a feature not in our table. What would k-means have returned, and how would you have known it was wrong?

### 2.3 Where k-means fails

k-means minimizes squared distance to a centre, and that single fact determines the shapes it can find. Clusters that are round and similarly sized: fine. Anything else: trouble.

To see the failure cleanly we use **deliberately constructed** data rather than a real dataset, because we want a shape chosen to break the assumption.

In [ ]:
# RUN-TOGETHER
from sklearn.datasets import make_moons

X_moons, _ = make_moons(n_samples=300, noise=0.06, random_state=210)
km_moons = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.scatterplot(x=X_moons[:, 0], y=X_moons[:, 1], color='gray', s=20, ax=axes[0])
axes[0].set_title('the data: two crescents')
sns.scatterplot(x=X_moons[:, 0], y=X_moons[:, 1], hue=km_moons.labels_, palette='viridis', s=20, ax=axes[1], legend=False)
axes[1].set_title('what k-means finds')
plt.show()

Your eye sees two crescents. k-means draws a straight line through both of them, because the only clustering it can express is "closest centre wins", and that always carves space into straight-edged regions.

**The three standing limitations of k-means:**

1. You must choose `k` in advance (Section 3).
2. The result depends on the initial centroids (Activity 9.1, part C).
3. It assumes clusters are round and comparably sized.

Section 5 introduces an algorithm with a different assumption, which handles the crescents easily and fails elsewhere. There is no clustering algorithm that always wins.

---
## 3. Choosing k
---

If `k` must be chosen in advance, how do we choose it?

The honest answer is that domain knowledge usually decides: a marketing team that can run three campaigns wants three segments. But there is a useful diagnostic, and it reuses the WCSS you computed by hand.

Run k-means for many values of `k` and plot WCSS against `k`, using the penguins again. WCSS always falls as `k` rises (more centroids means everything is nearer to one), so we are not looking for a minimum. We are looking for the point where the *improvement* stops being worth it: a bend, or **elbow**.

In [ ]:
# RUN-TOGETHER
ks = range(1, 9)
wcss = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_pen).inertia_ for k in ks]

for k, w in zip(ks, wcss):
    print(f'k = {k}   WCSS = {w:7.1f}')

sns.lineplot(x=list(ks), y=wcss, marker='o')
plt.title('Elbow plot: penguins, four scaled measurements')
plt.xlabel('k (number of clusters)')
plt.ylabel('WCSS (inertia)')
plt.xticks(list(ks))
plt.show()

#### **Activity 9.2 - Read the elbow**

**A.** Look at the printed WCSS values. Compute the *drop* at each step: from k=1 to k=2, from k=2 to k=3, from k=3 to k=4. Which drop is by far the largest?

**B.** Based on the drops alone, what value of `k` does the elbow suggest? We know the data contains **three** species. Does the elbow agree?

**C.** This disagreement is not a mistake in the plot. Using what you learned in Section 2.2 about *what* k-means actually separated, explain why the elbow points where it does.

**D.** WCSS at `k = 8` is lower than at any smaller `k`. Explain in one sentence why that does not make 8 the better choice. What would WCSS be if `k` equalled the number of penguins?

**E.** Now build the same elbow plot for the crescent data `X_moons`. Is there a clear elbow? What does its absence tell you?

In [ ]:
# FILL-IN  (Activity 9.2, Part D)
wcss_moons = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(____).inertia_ for k in ks]

sns.lineplot(x=list(ks), y=wcss_moons, marker='o')
plt.title('Elbow plot for the crescent data')
plt.xlabel('k'); plt.ylabel('WCSS')
plt.show()

---
## 4. Hierarchical Clustering
---

k-means asks you to commit to `k` before you see anything. **Hierarchical clustering** builds the entire family of clusterings at once, from "every point alone" up to "everything in one group", and lets you cut wherever you like afterwards.

The version we use is **agglomerative** (bottom-up):

1. Start with every point as its own cluster.
2. Merge the two **closest** clusters.
3. Repeat until one cluster remains, recording the distance at each merge.

Step 2 needs a definition of distance *between clusters*, not between points, and there is more than one reasonable choice. That choice is called the **linkage**.

**Definition.** Under **single linkage**, the distance between two clusters is the distance between their two *nearest* members:

$$d(A, B) \;=\; \min_{a \in A,\; b \in B} \; \lvert a - b \rvert$$

The record of merges is drawn as a **dendrogram**: a tree whose branch heights are the distances at which merges happened. Cutting the tree at a height gives you a clustering.

#### **Activity 9.3 - Build a dendrogram by hand**

Five points on a line:

$$A = 1, \quad B = 2, \quad C = 5, \quad D = 9, \quad E = 11$$

**Step 1. Fill in the distance matrix.** (It is symmetric, so only the upper triangle is needed.)

| | A | B | C | D | E |
| --- | --- | --- | --- | --- | --- |
| **A** | 0 |  |  |  |  |
| **B** | | 0 |  |  |  |
| **C** | | | 0 |  |  |
| **D** | | | | 0 |  |
| **E** | | | | | 0 |

**Step 2. Merge, four times.** Each round: find the smallest distance in the table, merge that pair, and record the height. Then recompute distances from the new cluster to everything else using **single linkage** (the minimum).

| merge # | clusters joined | height |
| --- | --- | --- |
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| 4 |  |  |

**Step 3. Sketch the dendrogram** with height on the vertical axis.

**Questions.**

**A.** Cut your dendrogram at height 3.5. Which clusters do you get?

**B.** Cut it at height 1.5 instead. How many clusters now?

**C.** k-means needed `k` before starting. What did hierarchical clustering need instead, and when did you need to supply it?

**D.** Under single linkage, the distance from $\{A,B\}$ to $C$ was the *minimum* of $d(A,C)$ and $d(B,C)$. Suppose we had used the **maximum** instead (called complete linkage). Redo merge 2 and say whether anything changes.

#### **Class Example 9.2 - The same five points, in code**

In [ ]:
# RUN-TOGETHER
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

pts = np.array([[1.], [2.], [5.], [9.], [11.]])
labels = ['A', 'B', 'C', 'D', 'E']

Z = linkage(pts, method='single')

dendrogram(Z, labels=labels)
plt.title('Single-linkage dendrogram')
plt.ylabel('merge height (distance)')
plt.show()

print('merge heights:', [round(z[2], 2) for z in Z])
print('cut at 3.5 ->', fcluster(Z, 3.5, criterion='distance'))

Check the printed merge heights against your table, and the cut against your answer to question A.

Notice what the dendrogram gives you that k-means never did: **every** clustering, at once. You choose the number of clusters by choosing a height, after looking at the picture rather than before.

---
## 5. DBSCAN: Clustering by Density
---

k-means asks "which center is nearest?" DBSCAN asks a different question: **"is this point in a crowded neighbourhood?"**

**Definition.** DBSCAN takes two parameters:

- `eps` ($\varepsilon$, "epsilon"): a radius defining what "nearby" means.
- `min_samples`: how many points must lie within that radius for a neighbourhood to count as crowded.

Every point is then labelled:

- a **core point** has at least `min_samples` points within `eps`;
- a **border point** is within `eps` of a core point but is not itself core;
- a **noise point** is neither.

Clusters are built by chaining core points together, so a cluster can be any shape at all: a crescent, a ring, a snake. And unlike k-means, DBSCAN is allowed to say *"this point belongs to nothing"*, which no version of k-means can do.

Watch it on the data that defeated k-means.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/09-unsupervised_learning/fig_dbscan_sketch.png?raw=true" width="640">

*Core points sit in crowded neighbourhoods, border points touch a core point, and noise points sit alone. Clusters grow by chaining core points, which is why a cluster can bend around a curve.*

In [ ]:
# RUN-TOGETHER
db = DBSCAN(eps=0.25, min_samples=5).fit(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.scatterplot(x=X_moons[:, 0], y=X_moons[:, 1], hue=km_moons.labels_, palette='viridis', s=20, ax=axes[0], legend=False)
axes[0].set_title('k-means, k = 2')
sns.scatterplot(x=X_moons[:, 0], y=X_moons[:, 1], hue=db.labels_, palette='viridis', s=20, ax=axes[1], legend=False)
axes[1].set_title('DBSCAN, eps = 0.25')
plt.show()

print('clusters found :', len(set(db.labels_)) - (1 if -1 in db.labels_ else 0))
print('points labelled noise:', (db.labels_ == -1).sum())

DBSCAN separates the crescents, and it was never told there were two. The number of clusters is an *output*, not an input.

The cost is that `eps` and `min_samples` now carry the difficulty that `k` used to. There is no free lunch: every clustering algorithm requires you to say what "similar" means, and each one asks in its own language.

## The No Free Lunch Theorem
> The No Free Lunch (NFL) theorem states that when performance is averaged across all possible problems, no single optimization or machine learning algorithm is universally superior. If an algorithm performs exceptionally well on one class of problems, it must pay for that advantage by performing worse on other problems

#### **Activity 9.4 - Tuning eps**

**A.** Predict: what happens to the number of clusters if `eps` is very small? Very large?

**B.** Run the cell below for `eps` values 0.1, 0.25, and 1.0 and record the number of clusters and the number of noise points for each.

**C.** Which of the three would you report, and why?

**D.** In one sentence: DBSCAN found the crescents and k-means could not. Name a dataset shape where you would expect the opposite.

In [ ]:
# FILL-IN  (Activity 9.4, Part B)
for eps in [____, ____, ____]:
    db = DBSCAN(eps=eps, min_samples=5).fit(X_moons)
    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    print(f'eps={eps:5}  clusters={n_clusters}  noise={(db.labels_ == -1).sum()}')

---
## 6. Dimension Reduction with PCA
---
### 6.1 Why reduce dimensions?

The superstore data (https://www.kaggle.com/code/arshmankhalid/personalities-of-customers-discovering-consumer) has six columns recording how much each customer spent on wines, fruit, meat, fish, sweets, and gold. Six dimensions cannot be plotted, and Module 5's pairplot would need 15 panels.

There is a deeper problem than plotting, usually called the **curse of dimensionality**: as the number of features grows, points spread out, distances between them become more and more alike, and "nearest neighbour" stops meaning much. Every algorithm in this module rests on distance, so this matters.

**Principal Component Analysis (PCA)** replaces the original features with new ones, called **principal components**, built as combinations of the originals. The first component is chosen to capture as much of the variation in the data as possible; the second captures as much of what is left as possible, and so on. Keeping the first two or three often preserves most of the information while making the data drawable.

**One important rule: PCA must be given scaled data.** PCA chases variance, and variance depends on units. `Income` measured in dollars has a variance thousands of times larger than `NumWebPurchases`, so without scaling the first component would simply *be* income. Standardizing every column to mean 0 and standard deviation 1 puts them on equal footing.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/Notes/09-unsupervised_learning/fig_dimensionality_reduction_techniques.png?raw=true" width="620">

*PCA is one route through a larger family. We use it because it is the most widely applied, and because has a really cool connection to Linear Algebra.*

In [ ]:
# RUN-TOGETHER
superstore = pd.read_csv('https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/csv_data/superstore_data.csv')
print('shape:', superstore.shape)

spend_cols = ['MntWines', 'MntFruits', 'MntMeatProducts',
              'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
spend = superstore[spend_cols]
spend.describe().round(1)

### 6.2 Scale, then reduce

In [ ]:
# RUN-TOGETHER
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaled = StandardScaler().fit_transform(spend)   # mean 0, sd 1 for every column

pca = PCA(n_components=2).fit(scaled)
components = pca.transform(scaled)

print('variance explained by PC1:', round(pca.explained_variance_ratio_[0], 3))
print('variance explained by PC2:', round(pca.explained_variance_ratio_[1], 3))
print('the two together         :', round(pca.explained_variance_ratio_[:2].sum(), 3))

Two numbers now stand in for six, and they retain most of the variation. What do they *mean*? Read the **loadings**: how much each original feature contributes to each component.

In [ ]:
# RUN-TOGETHER
loadings = pd.DataFrame(pca.components_.T, index=spend_cols, columns=['PC1', 'PC2']).round(2)
print(loadings)

> **Discuss.** Look at the PC1 column. Are the loadings all the same sign, or mixed? If a customer scores high on PC1, what does that say about their spending? Now do the same for PC2, where the signs are more interesting.
>
> A principal component is not handed to you with a name. Naming it from its loadings is interpretation, and it is where the substantive-expertise circle from Module 1 is needed.

In [ ]:
# RUN-TOGETHER
# The six spending columns, drawn in two dimensions.
sns.scatterplot(x=components[:, 0], y=components[:, 1], s=8, alpha=0.4)
plt.title('Superstore customers in two principal components')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.show()

#### **Activity 9.5 - PCA and clustering together**

The two halves of this module combine naturally: reduce first, then cluster in the reduced space.

**A.** Run k-means with `k = 3` on `components` (the two-column PCA output) and colour the scatterplot by cluster.

**B.** Build an elbow plot for this data. Does it suggest 3?

**C.** Redo part A on the **unscaled** `spend` data instead of `scaled`. Compare the pictures. Which single original column do you think is dominating, and why?

**D.** In two sentences: what would you tell the store's marketing team? Name the number of customer groups you found, and one honest caution about that number.

In [ ]:
# FILL-IN  (Activity 9.5, Part A)
km_pca = KMeans(n_clusters=____, n_init=10, random_state=0).fit(components)

sns.scatterplot(x=components[:, 0], y=components[:, 1], hue=____, s=8, alpha=0.6, palette='viridis', legend=False)
plt.title('Superstore customers, clustered in PCA space')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.show()

---
## 7. Choosing an Algorithm
---

| | k-means | Hierarchical | DBSCAN |
| --- | --- | --- | --- |
| Number of clusters | you choose `k` first | choose a cut height afterwards | an output, not an input |
| Cluster shapes | round, similar sizes | depends on linkage | any shape |
| Can label a point "none" | no | no | yes, as noise |
| Scales to large data | well | poorly | moderately |
| Main difficulty | choosing `k`, initial centroids | choosing linkage and cut | choosing `eps` |

No Free Lunch

**CAUTION.** All three algorithms will return an answer on any dataset you hand them, including data with no group structure at all. k-means asked for three clusters will produce three clusters from pure noise, and the picture will look convincing. There is no error message for "this data has no clusters".

## Suggested Exercises

1. Six points lie on a line at $x = 2,\ 3,\ 4,\ 10,\ 11,\ 12$, and you run k-means with $k = 2$ starting from $c_1 = 2$ and $c_2 = 3$.

    a. Carry out the assign-and-update steps by hand until convergence, recording the WCSS after each update.

    b. How many iterations were needed?

    c. Compute the WCSS of the clustering $\{2, 3, 4, 10\}$, $\{11, 12\}$. Is it larger or smaller than what k-means found? What does that tell you about the answer k-means reached?

    d. Explain why WCSS can never increase from one iteration to the next.

2. A dendrogram built by single linkage on points P, Q, R, S shows merges at heights 1 (P and Q), 2 (R and S), and 6 (the two pairs).

    a. How many clusters do you get cutting at height 1.5? At 4? At 7?

    b. The last merge happens at height 6 while the first two happen at 1 and 2. What does that gap suggest about the structure of this data?

    c. You are told the true distance from Q to R is 6.5. Is that consistent with the dendrogram under single linkage? Explain.

    d. Give one advantage of hierarchical clustering over k-means, and one disadvantage.

3. A colleague runs DBSCAN on 5,000 sensor readings with `eps=0.5, min_samples=10` and reports "one cluster of 4,900 points, plus 100 noise points."

    a. Give two different explanations for a single enormous cluster.

    b. Which parameter would you change first, and in which direction?

    c. Your colleague suggests switching to k-means with `k=4` instead, "so that nothing gets thrown away as noise." Give one reason this could be the wrong fix.

    d. Name one situation where labelling points as noise is the feature you actually want.

4. You run PCA on eight columns of student data and find PC1 explains 78% of the variance, with loadings that are large and positive on all four exam-score columns and near zero on the other four.

    a. Propose a name for PC1 and justify it from the loadings.

    b. Someone asks whether PC1 "is" the student's average exam score. Is that right? Explain the relationship carefully.

    c. You forgot to scale before running PCA. One column was recorded in dollars and ranged from 0 to 40,000. Predict what PC1 would have looked like, and why.

    d. PC1 through PC3 explain 94% of the variance. State one thing you gain and one thing you lose by working with those three columns instead of the original eight.